# 랭체인으로 RAG 시작하기

* Langchain으로 RAG를 구현하기 실습
* Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
root_dir = '/content/drive/MyDrive/Colab Notebooks/Aiffel/20260316-RAG'

In [7]:
!pip install U -q langchain-community langchain-core langchain_openai langchain_chroma langchain

In [8]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [11]:
!pip install -qqq pypdf pdf2image docx2txt pdfminer unstructured

### Step 1 : Document Loaders  

Document Loader : Raw Data -> Document 객체(text + metadata)

* PDF, 웹페이지, CSV 등 -> 파싱 -> Chunking·Embedding·검색(Retrieval) 단계에서 사용
* RAG 파이프라인 첫 단계
* 참조 : https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader
* 각 페이지 당 하나의 Document 로 변환


In [10]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(f"{root_dir}/data/Demian.pdf")
pages = loader.load_and_split()

In [21]:
import pprint
import json

print("--- metadata ---")
print(json.dumps(pages[0].metadata, indent=2))
print("\n")

print("--- page_content ---")
print(pages[0].page_content)

--- metadata ---
{
  "producer": "Adobe Acrobat Standard DC 19 Paper Capture Plug-in",
  "creator": "ScanFix(TM) Enhanced",
  "creationdate": "2015-09-10T01:40:29+00:00",
  "moddate": "2019-01-30T17:47:47+01:00",
  "source": "/content/drive/MyDrive/Colab Notebooks/Aiffel/20260316-RAG/data/Demian.pdf",
  "total_pages": 182,
  "page": 0,
  "page_label": "1"
}


--- page_content ---
DEMIAN 
• 
Downloaded from https://www.holybooks.com


In [22]:
print(json.dumps(pages[10].metadata, indent=2))
print(pages[10].page_content)

{
  "producer": "Adobe Acrobat Standard DC 19 Paper Capture Plug-in",
  "creator": "ScanFix(TM) Enhanced",
  "creationdate": "2015-09-10T01:40:29+00:00",
  "moddate": "2019-01-30T17:47:47+01:00",
  "source": "/content/drive/MyDrive/Colab Notebooks/Aiffel/20260316-RAG/data/Demian.pdf",
  "total_pages": 182,
  "page": 10,
  "page_label": "11"
}
TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples do

#### CSVLoader

* 각 행이 하나의 Document 객체로 변환

In [27]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(f"{root_dir}/data/titanic.csv")

data = loader.load()

In [28]:
print(data[:2])

r = json.dumps(data[0].metadata, indent=2)
print(r)

r = data[0].page_content
print(r)

[Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/Aiffel/20260316-RAG/data/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'), Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/Aiffel/20260316-RAG/data/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C')]
{
  "source": "/content/drive/MyDrive/Colab Notebooks/Aiffel/20260316-RAG/data/titanic.csv",
  "row": 0
}
PassengerId: 1
Survived: 0
Pclass: 3
Name: Braund, Mr. Owen Harris
Sex: male
Age: 22
SibSp: 1
Parch: 0
Ticket: A/5 21171
Fare: 7.25
Cabin: 
Embarked: S


#### 웹베이스로더  
웹페이지 텍스트 -> Document 객체

In [29]:
from langchain_community.document_loaders import WebBaseLoader

In [30]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

# print(documents[0].page_content)

### Step2 : TextSplitters
* 텍스트 문서 -> Chunk로 분해
* 각 Chunk -> 1:1로 Embedding되어 VectorStore에 저장
* Chunk 단위 : RAG 시스템에서 검색 응답의 기본 단위



In [31]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

* CharacterTextSplitter
    * 하나의 고정된 구분자(separator)로 텍스트 분할
    * 단순하고 직관적
    * Chunk가 토큰 제한을 초과하는 경우가 발생 가능성
    
* RecursiveCharacterTextSplitter
    * 여러 구분자를 순차 적용, 텍스트를 재귀적으로 분할.
        * 줄바꿈, 문장 구분자, 구두점 등
    * 토큰 제한 만족에 유리
    * 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있는 위험
    
* 상황에 맞게 잘 선택하기

In [33]:
with open(f"{root_dir}/data/state_of_the_union.txt") as f:
    text = f.read()

In [34]:
#chunk_overlap : chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정

text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

In [37]:
print(len(chunks))

print(chunks[0])
print(len(chunks[0]))

print(chunks[1])
print(len(chunks[1]))

42
Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.
939
Groups of citizens blocking tanks with their bodies. 

각 chunk의 길이 확인

In [38]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할
  
* 문자 수, 단어 수를 기준으로 텍스트 분할 시 모델의 입력 토큰 제한이 초과 될 수 있음.

* 실제 서비스 에서는 토큰 단위로 텍스트를 분할하는 방식 적용
* 영어는 단어 수 > 토큰 수, 한국어는 단어 수 < 토큰 수 일 수 있다.

In [40]:
!pip -q install tiktoken

In [42]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

In [43]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


### Step3 : TextEmbedding  
* 텍스트 -> 수치 벡터(vector) 형태로 변환 과정
* Document : 임베딩 벡터 = 1:1로 매칭됨.
* 의미적 유사성을 반영하도록 설계 되어 있음.

* 변환된 벡터는 VectorStore에 저장
* 질의(Query) 벡터와의 유사도 계산으로 의미적으로 가까운 문서를 검색하는 데 사용

공식 문서는 아래 링크에서 확인할 수 있습니다.
https://ai.google.dev/docs/embeddings_guide?hl=ko

In [54]:
from langchain_openai import OpenAIEmbeddings

In [55]:
# https://developers.openai.com/api/docs/guides/embeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [48]:
!curl ipinfo.io

{
  "ip": "136.119.54.170",
  "hostname": "170.54.119.136.bc.googleusercontent.com",
  "city": "Council Bluffs",
  "region": "Iowa",
  "country": "US",
  "loc": "41.2619,-95.8608",
  "org": "AS396982 Google LLC",
  "postal": "51502",
  "timezone": "America/Chicago",
  "readme": "https://ipinfo.io/missingauth"
}

In [49]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

In [53]:
print(len(embeddings)) # 3

print(len(embeddings[0])) # 1536 차원
print(len(embeddings[1]))
print(len(embeddings[2]))

3
1536
1536
1536


In [ ]:
len(embeddings[1])

1536

새로운 쿼리와 임베딩끼리 유사도를 계산

In [56]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [58]:
query = ["this is red fruit"]

In [59]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7477942151308524
0.48995458116791124
0.4084112226829356


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
* Embedding vector 저장소
* 벡터 간의 유사도 계산,
* 검색 인덱싱 구조

* 대표 VectorStore : Chroma, FAISS, ChromaDB

In [60]:
|!pip install -q chromadb

In [61]:
!pip install -q langchain-chroma

In [62]:
from langchain_community.vectorstores import Chroma

In [63]:
# from langchain.vectorstores import Chroma
# from langchain_chroma import Chroma

In [64]:
#!pip install --upgrade opentelemetry-api
#!pip install --upgrade opentelemetry-sdk

In [65]:
#from langchain_chroma import Chroma

DATA 준비

In [66]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader(f"{root_dir}/data/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [67]:
!pip show chromadb

Name: chromadb
Version: 1.5.5
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: langchain-chroma


Chroma에 임베딩  

In [70]:
pip install -q --upgrade opentelemetry-api opentelemetry-sdk

In [71]:
!pip install -q --upgrade chromadb langchain-chroma

In [72]:
db = Chroma.from_documents(docs, embedding_model)

쿼리 전송 -> Face, features, looks like 등 query와 연관된 페이지 출력

In [73]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [74]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

### Step5 : Retriever  

* 쿼리 -> Embedding 벡터로 변환
* VectorStore에 저장된 문서 벡터들과 비교
* 의미적으로 가장 유사한 문서(Chunk)를 찾아 반환

In [75]:
# from langchain.chains import RetrievalQA
from langchain_classic.chains import RetrievalQA


* 긴 문서 전체를 한 번에 LLM에 전달하는 대신,
* Retriever와 LLM을 결합한 RetrievalQA 체인으로
    * 문서에서 질문과 관련된 부분만 검색하고, 그 결과를 바탕으로 답변을 생성
    
* 길이가 긴 문서에서도 토큰 제한 지키며 근거 기반 질의응답 할 수 있다.

In [76]:
from langchain_openai import ChatOpenAI
from langchain_core.callbacks import StreamingStdOutCallbackHandler

llm = ChatOpenAI(model="gpt-4o")

체인의 종류와 검색(Retrieval) 방식, 파라미터 설정

* Retriever 전략
    * MMR(Maximal Marginal Relevance): 쿼리와의 유사도 + 문서 간 중복 제거, 다양성 확보하는 재정렬(Re-ranking) 전략

In [79]:
qa = RetrievalQA.from_chain_type(
    llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_type="mmr",
                              search_kwargs={"k": 3, "fetch_k" : 10}),
    return_source_documents=True)

* chain_type="stuff"
    * 검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식
    * 구조가 단순하고 이해하기 쉬움,
    * 문서 수가 많으면 토큰 사용량이 빠르게 증가
    * 초기 검증 단계는 stuff -> 문서 수가 많아지면 map_reduce나 refine 방식
    
* retriever
    * VectorStore에서 어떤 문서를 검색할지 결정하는 모듈
    
* search_type="mmr"
    * MMR Maximal Marginal Relevance
        * 쿼리 유사도, 문서 간 중복 감소, Re-ranking 전략
        
* search_kwargs={"k": 3, "fetch_k": 10}  
    - fetch_k VectorStore에서 우선적으로 가져올 후보 문서 개수
        - Re-ranking 이전 단계에서 사용됩니다.
    - k  
        - 최종 LLM에게 전달할 문서(Chunk)의 개수
        
    - 보통 fetch_k > k 로 설정, 좋은 품질 문서만 선별하는 방식

* return_source_documents=True
    - 답변 생성에 사용된 원문 문서(Chunk)를 함께 반환
    - 답변의 출처 표시, 검색 품질 디버깅
    - RAG 성능 평가

In [80]:
 query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_2905/3481366020.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = qa(query)


마크다운 형식으로 출력

In [82]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian's appearance is described as being different and unique. He has a face that is neither distinctly masculine nor childish, neither old nor young but almost timeless, bearing marks of other periods in history. This gives him a distinctive, almost otherworldly aura. His expression tends to show deep, quiet, almost fanatical yet passionate absorption. At times, his features appear similar to a female element, and there is something very compelling and enigmatic about his presence that makes him stand out from others.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [83]:
llm2 = ChatOpenAI(model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))

If you're referring to the character from Hermann Hesse's novel "Demian," the book does not provide a detailed physical description of Demian, leaving much to the reader’s imagination. Instead, the focus is more on his enigmatic and influential personality. Demian is portrayed as having an intense, penetrating presence and a mature, almost otherworldly aura that makes a strong impression on the protagonist, Emil Sinclair. The reader learns more about Demian through his philosophical insights and the impact he has on Sinclair’s spiritual and personal growth rather than through concrete physical attributes.

In [86]:
!pip install -q openai

In [85]:
import openai
from IPython.display import Markdown, display
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=OPENAI_API_KEY)

query = "how demian looks like"

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": query}
    ],
    temperature=0.0
)

openai_response_content = response.choices[0].message.content
display(Markdown(openai_response_content))

If you're referring to the character Demian from Hermann Hesse's novel "Demian," he is depicted as a mysterious and charismatic figure. Demian is often described as having a striking and intense presence, with features that convey both wisdom and a sense of otherworldliness. He is portrayed as being mature beyond his years, with penetrating eyes that seem to see through people. His appearance and demeanor reflect his role as a guide and mentor to the protagonist, Emil Sinclair, helping him navigate the complexities of self-discovery and spiritual awakening. The novel doesn't provide an exhaustive physical description, allowing readers to imagine Demian's appearance in a way that complements his enigmatic personality.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
다음은 Lagnchain으로 구현된 Question-Answer RAG 완성 예제입니다  


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q langchain langchain-google-genai chromadb pypdf sentence_transformers tiktoken

In [ ]:
!pip install U -q langchain-community langchain-core

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

root_dir = '/content/drive/MyDrive/Colab Notebooks/Aiffel/20260316-RAG'

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.vectorstores import Chroma

Text splitter 사용을 위한 준비입니다

In [87]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)

    return len(tokens)

### Step 1 Document loader

In [89]:
loader = PyPDFLoader(f"{root_dir}/data/Demian.pdf")
pages = loader.load_and_split()

### Step 2 Text splitters

In [90]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=5000, chunk_overlap=50, length_function = tiktoken_len)
texts = text_splitter.split_documents(pages)

### Step 3 Vector Empeddings

In [94]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [95]:
docsearch = Chroma.from_documents(texts, embedding_model)

### Step 4 Retrievers

In [101]:
from langchain_core.callbacks import StreamingStdOutCallbackHandler
llm = ChatOpenAI(model="gpt-4o")

In [103]:
qa = RetrievalQA.from_chain_type(
    llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_type="mmr",
                              search_kwargs={"k": 3, "fetch_k" : 10}),
    return_source_documents=True)

### Question Answering

In [104]:
query = "how demian looks like"
result = qa(query)

In [105]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having an expression and presence that is strikingly different from others. His face is described as being neither strictly masculine nor childish; it has a unique, almost timeless quality that seems to transcend specific periods of history. This face might be seen as neither solely a man's nor a boy's; instead, it possesses elements that are both elegant and otherworldly, possibly with a feminine element.

He is likened to an animal or a spirit, and his presence is often associated with an aura of quiet emptiness and celestial space. In certain moments, Demian seems almost like a stone idol, cold yet carrying an alarming secret life within. This description indicates his mysterious, enigmatic appearance and aura that fascinates and sometimes unsettles those around him.